# ReBRAC Broad Validation v2 — N2' Asymmetric-Critic Ablation

> 文档锚点：[`docs/rebrac_broad_validation_v2_plan.md`](../docs/rebrac_broad_validation_v2_plan.md) §9 backlog「AsymCritic ablation on N2'」· [`docs/rebrac_broad_validation_v2_report.md`](../docs/rebrac_broad_validation_v2_report.md) §4/§6.3 · [CLAUDE.md §3 Asymmetric Critic](../CLAUDE.md)
>
> 模板：[`notebooks/rebrac_c1_asym_critic_ablation.ipynb`](rebrac_c1_asym_critic_ablation.ipynb)（同款 ReBRAC + asym ablation）+ [`notebooks/rebrac_broad_validation_v2_core_completed_new.ipynb`](rebrac_broad_validation_v2_core_completed_new.ipynb)（N2' vanilla baseline 实跑命令）

## 前传 — 已知事实（vanilla N2'，broad-val v2 report §3）

| run | dataset (collector) | critic | seeds | test success | progress_ratio | termination |
|---|---|---|---:|---:|---:|---|
| **vanilla N2'** | privileged @ cross_u15/Re250 (oracle 70%) | **sym (s0 only)** | 42, 0 | **0.000 / 0.000** | 0.04 / −0.005 | 纯 OOB / OOB+timeout |
| online §7.6 floor | — (vanilla SAC s0) | — | — | **0.10** | — | catastrophic |
| oracle ceiling | privileged teacher (用 hull-integral flow 决策) | — | — | **0.70** | — | — |

vanilla N2' 中 **actor 和 critic 都只看 s0**，0% 失败有两个无法区分的解释：(a) **actor-fundamental** — actor 物理上没法从 s0 还原 privileged 决策规则；(b) **critic-fundamental** — critic 也只看 s0，价值估计烂 → TD target 烂 → 训练塌掉。

## 假设 — privileged critic 能否区分这两者？

CLAUDE.md §3 主方法论：**actor 受 deployment-realistic s0 约束，critic 训练时拿 hull-integral flow `privileged_obs=[u_eq,v_eq]` (dim=2) 算 TD target**；deployment 时 actor 仍只 s0。

本 notebook 把 vanilla N2' 重跑一遍，**唯一变量 = `--use-asymmetric-critic`**（critic 看 s0+priv，actor 仍 s0）。数据通路已就绪：privileged dataset 含 `privileged_obs` / `next_privileged_obs` 列（§3 硬检查）。

## Verdict 逻辑（事先 commit）

校准基线：vanilla N2' = 0.000、online floor = 0.10、oracle ceiling = 0.70。

| asym N2' success | verdict | 含义 |
|---:|---|---|
| **≤ 0.10**（≈ vanilla / ≤ online floor） | **ACTOR-FUNDAMENTAL 确认** | privileged critic 救不动 s0 actor → 天花板在 actor 的 s0→action 映射。**HARDENS** broad-val claim。（预期最可能 + 最强 paper 结果） |
| **0.10 – 0.40** | **MIXED ceiling** | privileged critic 部分有效 → 天花板含 actor-representation + critic-estimation 两成分。claim 软化为「primarily actor-fundamental」。补 seed 43 + stratified bootstrap。 |
| **≥ 0.40** | **CRITIC-CONTRIBUTING / 重 frame** | privileged critic 大幅 bridge → vanilla 天花板主要来自 critic 价值估计失败。「actor-fundamental」表述需 power-aware re-eval（描述修正，**非 "refuted"**）。同时是 asym-critic 方法在 offline partial-obs 上的 POSITIVE cross-line 结果。 |

## 与 vanilla N2' 的隔离

| 维度 | vanilla N2' | **本 notebook** |
|---|---|---|
| dataset / manifest / objective | privileged_re250_u15cross / single_u15_cross_tgt15 / arrival_v2 | **完全一致** |
| β1, β2 | 4.0, 2.0 | **4.0, 2.0**（anchor 不动） |
| critic | sym (s0) | **asym (s0 + 2-D priv)** |
| actor update | n/a | `--privileged-actor-update-mode zeros`（actor 改进 zero-pad priv，mimic deployment） |
| seeds | 42, 0 | **42, 0**（配对，必须相同） |

唯一变量：`--use-asymmetric-critic`。

## 1. 环境 sanity check

In [1]:
!lscpu | head -8
print()
!nvidia-smi

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  12
On-line CPU(s) list:                     0-11
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz

Tue May 26 17:44:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |              

In [2]:
import torch
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device 0      : {torch.cuda.get_device_name(0)}')

PyTorch       : 2.10.0+cu128
CUDA available: True
Device 0      : NVIDIA L4


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. 配置

复用 vanilla N2' 的 dataset / manifest / anchor 超参，仅加 `--use-asymmetric-critic`。checkpoint / results 单独命名树，不与 vanilla N2' 碰撞。

In [4]:
import os
from pathlib import Path

# 注意：Drive 路径含空格（"Colab Notebooks"），IPython `!python {VAR}` 插值不加引号，
# 绝对路径会在空格处被 shell 拆断（--save-dir 报 unrecognized arguments）。
# 因此所有喂给 !python 的路径一律用「相对仓库根」的相对路径（%cd 已进入仓库根），
# 相对路径不含空格，彻底回避此 bug（与 rebrac_c1_asym_critic_ablation.ipynb 同策略）。

# ---- N2' cell（与 vanilla baseline 完全一致）----
DATASET  = 'offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz'
MANIFEST = 'benchmarks/single_u15_cross_tgt15.json'
REGIME_NOTE = 'critical (U=1.5 / Re=250)'

# ---- 与 vanilla N2' 严格相同的 anchor 超参 ----
PROBE_LAYOUT        = 's0'
HISTORY_LENGTH      = 4
TASK_GEOMETRY       = 'cross_stream'
TARGET_SPEED        = 1.5
OBJECTIVE           = 'arrival_v2'
NUM_EPOCHS          = 64
BATCH_SIZE          = 256
HIDDEN_DIM          = 256
NUM_HIDDEN_LAYERS   = 3
ACTOR_LR            = '3e-4'
CRITIC_LR           = '3e-4'
GAMMA               = 0.99
TAU                 = 0.005
ACTOR_PENALTY_COEF  = 4.0
CRITIC_PENALTY_COEF = 2.0
POLICY_NOISE        = 0.2
NOISE_CLIP          = 0.5
POLICY_FREQ         = 2
GRAD_CLIP_NORM      = 10.0
NORMALIZER_EPS      = '1e-3'

# ---- 唯一变量：asymmetric critic ----
PRIVILEGED_ACTOR_UPDATE_MODE = 'zeros'  # actor 改进 zero-pad priv (deployment-realistic, CLAUDE.md §3 默认)

# ---- seeds：与 vanilla baseline 配对，必须相同 ----
SEEDS = [42, 0]

# ---- eval（与 vanilla baseline evaluate_offline 命令逐字一致以保证配对）----
EVAL_EPISODES    = 100   # 受 manifest 30-ep 约束实跑 30；保持命令一致
EVAL_SEED        = 123
EVAL_NUM_WORKERS = 4

# ---- 路径：独立树，全部相对仓库根（不含空格，!python 插值安全）----
CKPT_ROOT     = Path('checkpoints/offline/rebrac/broad_validation_v2_n2p_asym')
RESULT_ROOT   = Path('results/offline/rebrac/broad_validation_v2_n2p_asym')
SUMMARIES_DIR = RESULT_ROOT / 'summaries'
SUMMARIES_DIR.mkdir(parents=True, exist_ok=True)

# ---- vanilla baseline（配对对照源）----
VANILLA_ROOT = Path('results/offline/rebrac/broad_validation_v2/N2p')

def ckpt_dir(seed):   return CKPT_ROOT / f'seed_{seed}'
def result_dir(seed): return RESULT_ROOT / f'seed_{seed}'

print('cwd          =', Path('.').resolve())
print('SEEDS        =', SEEDS)
print('DATASET      =', DATASET)
print('MANIFEST     =', MANIFEST)
print('CKPT_ROOT    =', CKPT_ROOT)
print('RESULT_ROOT  =', RESULT_ROOT)
print('VANILLA_ROOT =', VANILLA_ROOT)

cwd          = /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
SEEDS        = [42, 0]
DATASET      = offline_data/privileged_s0_h4_arrival_v2_re250_u15cross_fixdone_ep1000/transitions.npz
MANIFEST     = benchmarks/single_u15_cross_tgt15.json
CKPT_ROOT    = checkpoints/offline/rebrac/broad_validation_v2_n2p_asym
RESULT_ROOT  = results/offline/rebrac/broad_validation_v2_n2p_asym
VANILLA_ROOT = results/offline/rebrac/broad_validation_v2/N2p


## 3. Pre-flight — 验证 dataset 含 `privileged_obs` / `next_privileged_obs`

asym critic 在 TD target 与 critic loss 时取这两列；必须存在且 dim=2（body-frame `[u_eq, v_eq]`）。同时确认 vanilla baseline 已 sync 回仓库（配对对照需要）。

In [5]:
import numpy as np

npz_path = Path(DATASET)
assert npz_path.exists(), f'missing dataset: {npz_path}'
data = np.load(npz_path)
print('npz keys:', list(data.keys()))

for key in ('privileged_obs', 'next_privileged_obs'):
    assert key in data.files, f'MISSING column {key} — asym critic 必需此列'
    arr = data[key]
    assert arr.ndim == 2 and arr.shape[1] == 2, f'{key} shape={arr.shape} (期望 [N, 2])'
    print(f'  {key:20s} shape={arr.shape}  '
          f'mean=[{arr[:,0].mean():+.3f}, {arr[:,1].mean():+.3f}]  '
          f'std=[{arr[:,0].std():.3f}, {arr[:,1].std():.3f}]')

print()
for seed in SEEDS:
    vb = VANILLA_ROOT / f'seed_{seed}' / 'test_result.json'
    print(f'  vanilla baseline seed {seed}: '
          f'{"OK" if vb.exists() else "MISSING — 先 sync vanilla N2 回仓库"}  {vb}')

print('\n[OK] dataset asym-critic ready')

npz keys: ['obs', 'actions', 'next_actions', 'rewards', 'costs', 'next_obs', 'dones', 'terminateds', 'truncateds', 'terminal_reason_codes', 'behavior_policy_codes', 'privileged_obs', 'next_privileged_obs']
  privileged_obs       shape=(212783, 2)  mean=[-0.979, +0.004]  std=[0.613, 0.977]
  next_privileged_obs  shape=(212783, 2)  mean=[-0.985, +0.004]  std=[0.610, 0.973]

  vanilla baseline seed 42: OK  results/offline/rebrac/broad_validation_v2/N2p/seed_42/test_result.json
  vanilla baseline seed 0: OK  results/offline/rebrac/broad_validation_v2/N2p/seed_0/test_result.json

[OK] dataset asym-critic ready


## 4. 训练 — ReBRAC anchor (β1=4.0, β2=2.0) + asym critic × 2 seeds

新增 flag：
- `--use-asymmetric-critic` — critic 用 (s, a, priv_obs) 算 Q
- `--privileged-actor-update-mode zeros` — actor 改进时 zero-pad priv 通道（mimic deployment）

其余与 vanilla N2' 完全一致。预期 ~30–40 min/seed × 2 ≈ 1–1.5h L4。skip 判定用 `agent_final.pt`。

In [6]:
import time

for seed in SEEDS:
    cdir = ckpt_dir(seed)
    save_dir_str = str(cdir)
    if (cdir / 'agent_final.pt').exists():        # skip 判定用 agent_final.pt
        print(f'[skip] seed {seed} done: {cdir}')
        continue
    cdir.mkdir(parents=True, exist_ok=True)
    print(f'\n========== train seed {seed} (asym critic) → {cdir} ==========')
    t0 = time.time()
    !python -u -m scripts.train_offline \
        --algo rebrac \
        --offline-data {DATASET} \
        --manifest {MANIFEST} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --task-geometry {TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --sampling-mode shuffle_no_replacement \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-hidden-layers {NUM_HIDDEN_LAYERS} \
        --actor-lr {ACTOR_LR} \
        --critic-lr {CRITIC_LR} \
        --gamma {GAMMA} \
        --tau {TAU} \
        --actor-penalty-coef {ACTOR_PENALTY_COEF} \
        --critic-penalty-coef {CRITIC_PENALTY_COEF} \
        --policy-noise {POLICY_NOISE} \
        --noise-clip {NOISE_CLIP} \
        --policy-freq {POLICY_FREQ} \
        --grad-clip-norm {GRAD_CLIP_NORM} \
        --normalizer-eps {NORMALIZER_EPS} \
        --critic-layernorm \
        --no-actor-layernorm \
        --use-asymmetric-critic \
        --privileged-actor-update-mode {PRIVILEGED_ACTOR_UPDATE_MODE} \
        --eval-every 0 \
        --skip-final-eval \
        --log-every 1000 \
        --seed {seed} \
        --device cuda \
        --save-dir {save_dir_str}
    print(f'[done] seed {seed} ({(time.time()-t0)/60:.1f} min) — 若上方有 traceback 则训练失败')


========== train seed 42 (asym critic) → checkpoints/offline/rebrac/broad_validation_v2_n2p_asym/seed_42 ==========
[offline] algo=rebrac transitions=212783 obs_dim=48 action_dim=2 protocol=privileged-critic tensor_replay=on eval_workers=1 sampling=shuffle_no_replacement total_steps=53248
[train] step=1 epoch=1 epoch_step=1 q=-0.403 critic=87.171 actor=6.435 bc=1.361 lambda=2.465 critic_pen=1.416
[train] step=1000 epoch=2 epoch_step=168 q=-2.980 critic=206.633 actor=1.036 bc=0.078 lambda=0.243 critic_pen=0.181
[train] step=2000 epoch=3 epoch_step=336 q=-2.505 critic=54.810 actor=0.827 bc=0.105 lambda=0.162 critic_pen=0.171
[train] step=3000 epoch=4 epoch_step=504 q=-2.720 critic=5.554 actor=0.744 bc=0.112 lambda=0.109 critic_pen=0.207
[train] step=4000 epoch=5 epoch_step=672 q=-2.659 critic=214.020 actor=0.685 bc=0.108 lambda=0.096 critic_pen=0.172
[train] step=5000 epoch=7 epoch_step=8 q=-1.425 critic=41.204 actor=0.474 bc=0.093 lambda=0.073 critic_pen=0.180
[train] step=6000 epoch=8

## 5. Eval — 100-ep test manifest（实跑 30，manifest-bound）

`evaluate_offline` 自动从 `trainer_state.json` 读 `use_asymmetric_critic=True` 重建 `AsymmetricQNetwork`；**actor eval 时仍只看 s0**（不传 priv），与 deployment + vanilla baseline 完全一致。命令与 vanilla N2' eval 逐字相同（`--seed 123`，同 manifest）以保证配对。skip 判定用 `test_result.json`。

In [7]:
for seed in SEEDS:
    cdir = ckpt_dir(seed)
    rdir = result_dir(seed)
    test_json = rdir / 'test_result.json'
    test_json_str = str(test_json)
    if test_json.exists():
        print(f'[skip] eval done: {test_json}')
        continue
    if not (cdir / 'agent_final.pt').exists():
        print(f'[warn] missing agent_final.pt: {cdir} (训练未完成?)')
        continue
    rdir.mkdir(parents=True, exist_ok=True)
    print(f'\n========== eval seed {seed} (asym) → {test_json} ==========')
    !python -u -m scripts.evaluate_offline \
        --checkpoint {str(cdir)} \
        --agent-file agent_final.pt \
        --manifest {MANIFEST} \
        --episodes {EVAL_EPISODES} \
        --seed {EVAL_SEED} \
        --device cuda \
        --num-workers {EVAL_NUM_WORKERS} \
        --worker-device cpu \
        --output-json {test_json_str}


========== eval seed 42 (asym) → results/offline/rebrac/broad_validation_v2_n2p_asym/seed_42/test_result.json ==========
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 30
success_rate          : 0.0%
avg_return            : -243.60 +/- 45.46
avg_safety_cost       : 18.555 +/- 11.367
avg_time_s            : 73.80 +/- 46.75
avg_time_s_success    : nan
avg_energy            : 58127.81 +/- 37217.70
avg_path_length_m     : 80.30 +/- 44.54
avg_progress_ratio    : 0.143 +/- 0.414
avg_path_efficiency   : 0.146 +/- 0.246
termination           : {'out_of_bounds': 30}
benchmark_manifest    : benchmarks/single_u15_cross_tgt15.json

========== eval seed 0 (asym) → results/offline/rebrac/broad_validation_v2_n2p_asym/seed_0/test_result.json ==========
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 30
success_rate          : 0.0%
avg_return       

## 6. Raw JSON dump（诊断 — asym test_result.json 字段全表）

In [8]:
import json
for seed in SEEDS:
    test_json = result_dir(seed) / 'test_result.json'
    print(f'\n=== asym seed {seed} → {test_json} ===')
    if not test_json.exists():
        print('  MISSING'); continue
    d = json.loads(test_json.read_text())
    for k in sorted(d.keys()):
        v = d[k]
        if isinstance(v, (int, float, str, bool, type(None))):
            print(f'  {k}: {v}')
        elif isinstance(v, list):
            print(f'  {k}: list(len={len(v)})')
        elif isinstance(v, dict):
            print(f'  {k}: dict(keys={list(v.keys())})')


=== asym seed 42 → results/offline/rebrac/broad_validation_v2_n2p_asym/seed_42/test_result.json ===
  energy_cost_gain: 0.0
  eval_cost: 18.55490956221802
  eval_cost_std: 11.366915733349117
  eval_energy: 58127.80613501183
  eval_energy_std: 37217.70174724867
  eval_energy_success: None
  eval_episode_results: list(len=30)
  eval_manifest_flow_path: wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy
  eval_path_efficiency: 0.14569683209690235
  eval_path_efficiency_std: 0.24640809210669198
  eval_path_length_m: 80.29818326557675
  eval_path_length_m_std: 44.54146994882195
  eval_path_length_success: None
  eval_progress_ratio: 0.14301766445572778
  eval_progress_ratio_std: 0.41426488720210564
  eval_return: -243.5990249010858
  eval_return_std: 45.46015312339198
  eval_safety_cost: 18.55490956221802
  eval_safety_cost_std: 11.366915733349117
  eval_success_rate: 0.0
  eval_termination_counts: dict(keys=['out_of_bounds'])
  eval_time_s: 73.8033333333325
  eval_time_s_std

## 7. Vanilla vs Asym 配对对比 + 自动 verdict

加载 asym（本 notebook）与 vanilla N2'（broad-val v2 baseline）逐 seed 配对，对比 success / progress_ratio / termination 分布，应用 §0 事先 commit 的机制判别 gate。

In [9]:
import json
import statistics

ONLINE_FLOOR   = 0.10
ORACLE_CEILING = 0.70

def _f(x):
    return float('nan') if x is None else float(x)

def _load(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else None

def _term_pct(d):
    n = max(_f(d.get('num_eval_episodes', 30)), 1.0)
    t = d.get('eval_termination_counts', {})
    return (t.get('goal', 0) / n, t.get('out_of_bounds', 0) / n, t.get('timeout', 0) / n)

rows = []
for seed in SEEDS:
    a = _load(result_dir(seed) / 'test_result.json')
    v = _load(VANILLA_ROOT / f'seed_{seed}' / 'test_result.json')
    if a is None:
        print(f'[missing] asym seed {seed} — 跳过'); continue
    ag, ao, at = _term_pct(a)
    asym_succ    = _f(a.get('eval_success_rate'))
    vanilla_succ = _f(v.get('eval_success_rate')) if v else float('nan')
    rows.append({
        'seed': seed,
        'vanilla_succ': vanilla_succ,
        'asym_succ': asym_succ,
        'delta_succ': asym_succ - vanilla_succ,
        'vanilla_progress': (_f(v.get('eval_progress_ratio')) if v else float('nan')),
        'asym_progress': _f(a.get('eval_progress_ratio')),
        'asym_patheff': _f(a.get('eval_path_efficiency')),
        'asym_goal': ag, 'asym_oob': ao, 'asym_timeout': at,
    })

if not rows:
    print('[abort] no asym seeds completed; cannot verdict')
else:
    print('=' * 104)
    print(f'{"seed":>5}{"vanilla":>10}{"asym":>9}{"Δ(pp)":>9}'
          f'{"van_prog":>10}{"asym_prog":>11}   goal/oob/timeout (asym)')
    print('-' * 104)
    for r in rows:
        print(f'{r["seed"]:>5}{r["vanilla_succ"]:>10.4f}{r["asym_succ"]:>9.4f}'
              f'{r["delta_succ"]*100:>+8.2f}{r["vanilla_progress"]:>10.4f}{r["asym_progress"]:>11.4f}'
              f'   {r["asym_goal"]:.2f}/{r["asym_oob"]:.2f}/{r["asym_timeout"]:.2f}')
    print('=' * 104)

    asym_mean = statistics.mean(r['asym_succ'] for r in rows)
    asym_std  = statistics.stdev([r['asym_succ'] for r in rows]) if len(rows) > 1 else 0.0
    van_mean  = statistics.mean(r['vanilla_succ'] for r in rows)
    lift_pp   = (asym_mean - van_mean) * 100
    recovery  = asym_mean / ORACLE_CEILING * 100

    print(f"vanilla N2' (s0 critic)   mean = {van_mean:.4f}")
    print(f"asym    N2' (priv critic) mean = {asym_mean:.4f} +/- {asym_std:.4f}  ({len(rows)} seed)")
    print(f"Δ vs vanilla = {lift_pp:+.2f}pp   recovery of oracle 0.70 = {recovery:.1f}%   online floor = {ONLINE_FLOOR:.2f}")
    print('-' * 104)

    if asym_mean <= 0.10:
        verdict = 'ACTOR_FUNDAMENTAL_CONFIRMED'
        note = ("privileged critic 未能解锁 s0 actor (<= online floor) -> N2' 天花板在 actor 的 "
                "s0->action 映射，不在 critic 价值估计。HARDENS broad-val 'actor-fundamental "
                "partial-obs ceiling' claim：即便给 critic 完美 hull-integral flow，s0 actor 仍还原"
                "不了 privileged 决策规则。")
    elif asym_mean < 0.40:
        verdict = 'MIXED_CEILING'
        note = (f"privileged critic 部分抬升 ({lift_pp:+.2f}pp, recovery {recovery:.1f}%) -> 天花板含 "
                "actor-representation + critic-estimation 两成分。paper claim 软化为 'primarily "
                "actor-fundamental, with a measurable critic-value-estimation component'。"
                "建议补 seed 43 + (seed x episode) stratified bootstrap 收紧。")
    else:
        verdict = 'CRITIC_CONTRIBUTING__REFRAME'
        note = (f"privileged critic 大幅 bridge ({lift_pp:+.2f}pp, recovery {recovery:.1f}%) -> vanilla "
                "天花板主要来自 s0 critic 价值估计失败，而非 actor-fundamental。broad-val 'actor-"
                "fundamental' 表述需 power-aware re-evaluation (描述修正，非 'refuted')。"
                "同时这是 asymmetric-critic 方法的 POSITIVE 结果 — CLAUDE.md §3 privileged critic 在 "
                "offline partial-obs 上也能 bridge gap，可作 offline×online cross-line headline。")

    print(f'VERDICT: {verdict}')
    print(note)
    print('=' * 104)
    print("注：vanilla N2' = 0.000 是确定性 strong signal (非 no-power)；asym 端 2-seed std df=1 不稳，")
    print("    任何落入 MIXED / REFRAME 区的结果都应补 seed 43 + stratified bootstrap 再定论。")

    out = {
        'seeds': SEEDS,
        'online_floor': ONLINE_FLOOR, 'oracle_ceiling': ORACLE_CEILING,
        'vanilla_mean': van_mean, 'asym_mean': asym_mean, 'asym_std': asym_std,
        'lift_pp': lift_pp, 'recovery_of_oracle_pct': recovery,
        'per_seed': rows, 'verdict': verdict, 'note': note,
    }
    vp = SUMMARIES_DIR / 'asym_verdict.json'
    vp.write_text(json.dumps(out, indent=2))
    print(f'[wrote] {vp}')

 seed   vanilla     asym    Δ(pp)  van_prog  asym_prog   goal/oob/timeout (asym)
--------------------------------------------------------------------------------------------------------
   42    0.0000   0.0000   +0.00    0.0405     0.1430   0.00/1.00/0.00
    0    0.0000   0.0000   +0.00   -0.0050     0.0272   0.00/0.97/0.03
vanilla N2' (s0 critic)   mean = 0.0000
asym    N2' (priv critic) mean = 0.0000 +/- 0.0000  (2 seed)
Δ vs vanilla = +0.00pp   recovery of oracle 0.70 = 0.0%   online floor = 0.10
--------------------------------------------------------------------------------------------------------
VERDICT: ACTOR_FUNDAMENTAL_CONFIRMED
privileged critic 未能解锁 s0 actor (<= online floor) -> N2' 天花板在 actor 的 s0->action 映射，不在 critic 价值估计。HARDENS broad-val 'actor-fundamental partial-obs ceiling' claim：即便给 critic 完美 hull-integral flow，s0 actor 仍还原不了 privileged 决策规则。
注：vanilla N2' = 0.000 是确定性 strong signal (非 no-power)；asym 端 2-seed std df=1 不稳，
    任何落入 MIXED / REFRAME 区的结果都应补 seed 43 +

## 8. 跑完后清单（回到 local）

### Sync 回 git（只取 `results/`，checkpoint 大文件 gitignore）
```bash
rsync -av '<drive>/results/offline/rebrac/broad_validation_v2_n2p_asym/' \
  results/offline/rebrac/broad_validation_v2_n2p_asym/
```
包含 `seed_{42,0}/test_result.json` + `summaries/asym_verdict.json`。

### 落文档（按 verdict 分支）
- **任何 verdict**：[`docs/rebrac_broad_validation_v2_report.md`](../docs/rebrac_broad_validation_v2_report.md) §4 candidate A/B/C 段落填入实测；§6.3 backlog 行从「rev.4 candidate」改为「completed」。
- **ACTOR_FUNDAMENTAL_CONFIRMED**：ReBRAC paper（`paper/archive/rebrac_standalone/sections/discussion.tex` / `limitations.tex` / `appendix.tex` robustness）补一句「privileged critic ablation 确认 ceiling 为 actor-fundamental」；这是挡审稿人最强的一发。
- **MIXED / CRITIC_CONTRIBUTING**：先补 seed 43 + stratified bootstrap，再决定是否改 broad-val 主叙事；按用户偏好用「描述修正 + power-aware re-eval」框架，不写「refuted」。

### 若需 3-seed power
本 notebook + vanilla N2' 都加 seed 43 → 重跑（skip-resume 自动跳过 42 / 0），保持配对。